# 01 · A vision-language model from scratch

A VLM is three blocks glued together:

```
image ─► [1] vision encoder (ViT) ─► [2] connector/projector ─┐ visual tokens
                                                                ▼
text  ─► tokenizer ─► embeddings ──────────────────────────► merge ─► [3] decoder LM ─► next token
```

LLaVA, SmolVLM, Qwen-VL, Cosmos Reason and the VLM inside GR00T all follow this blueprint. They differ in size, resolution handling and training data. Here we build every piece of `TinyVLM` (`src/edge_vlm/model.py`) and look at the tensor shapes at each step. Everything in this notebook runs on a laptop CPU in seconds.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

import math, time
import torch, torch.nn.functional as F
import matplotlib.pyplot as plt

from edge_vlm.model import TinyVLM, TinyVLMConfig, pixel_shuffle, rope_tables, apply_rope
from edge_vlm.tokenizer import RobotTokenizer
from edge_vlm.sim import FetchWorld
from edge_vlm.data import image_to_tensor

torch.manual_seed(0)
world = FetchWorld(seed=1)
world.reset(target='red cup', target_room='kitchen')
img = world.render('kitchen')
plt.imshow(img); plt.title('robot camera in the kitchen (64x64)'); plt.axis('off'); plt.show()
print('objects here:', world.visible_objects('kitchen'))

## Step 1: patchify, turning an image into a sequence

Transformers eat sequences. A ViT cuts the image into $P\times P$ patches, flattens each one and projects it to a vector. A strided convolution does exactly this. Our 64x64 image with 8x8 patches gives $8\times8=64$ tokens.

*Real models:* SigLIP at 512 px with 16 px patches gives 1024 patches.

In [ ]:
x = image_to_tensor(img).unsqueeze(0)          # (1, 3, 64, 64), values in [-1, 1]
P = 8
patches = x.unfold(2, P, P).unfold(3, P, P)      # (1, 3, 8, 8, P, P)
patches = patches.permute(0, 2, 3, 1, 4, 5).reshape(1, 64, 3 * P * P)
print('manual patches:', tuple(patches.shape), '-> 64 tokens of 192 raw pixel values each')

cfg = TinyVLMConfig()
model = TinyVLM(cfg).eval()
tokens = model.vision.patch(x)
print('PatchEmbed (conv) output:', tuple(tokens.shape), '-> 64 tokens of dim', cfg.vision_dim)

fig, axes = plt.subplots(8, 8, figsize=(4, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(img[(i // 8) * P:(i // 8 + 1) * P, (i % 8) * P:(i % 8 + 1) * P]); ax.axis('off')
plt.suptitle('the 64 patches = 64 visual tokens'); plt.show()

## Step 2: the vision transformer

Each block does `x + Attention(LN(x))` and then `x + MLP(LN(x))`. Vision attention is **bidirectional**, so every patch can look at every other patch. That lets the model relate *red* pixels to a *cup* shape.

In [ ]:
with torch.no_grad():
    feats = model.vision(x)
print('vision features:', tuple(feats.shape))

## Step 3: pixel shuffle, the cheapest edge optimization

Every visual token costs LM compute and KV-cache memory. Pixel shuffle merges each $s\times s$ neighbourhood of tokens into **one** token with $s^2\times$ the channels. That is $s^2$ fewer tokens with no information thrown away. SmolVLM uses $s=4$ (1024 to 64 tokens). Qwen2-VL merges 2x2 the same way.

Below, each token carries its own (row, col) so you can see who gets grouped with whom.

In [ ]:
g = 4
coords = torch.tensor([[r, c] for r in range(g) for c in range(g)], dtype=torch.float).unsqueeze(0)
shuffled = pixel_shuffle(coords, 2)
print('before:', tuple(coords.shape), ' after:', tuple(shuffled.shape))
for i in range(4):
    print(f'new token {i} = patches', shuffled[0, i].view(4, 2).int().tolist())

print('\nTinyVLM: ', cfg.num_patches, 'patches ->', cfg.num_image_tokens, 'visual tokens')

## Step 4: the projector (connector)

The vision encoder and the LM were born in different embedding spaces. A small MLP maps each (shuffled) visual token into the LM's embedding space, so to the LM it looks like just another word. In two-stage VLM training, stage 1 often trains *only* this projector.

In [ ]:
with torch.no_grad():
    visual_tokens = model.projector(feats)
print('projector:', tuple(feats.shape), '-> pixel shuffle ->', (1, cfg.num_image_tokens, cfg.vision_dim * 4), '-> MLP ->', tuple(visual_tokens.shape))

## Step 5: tokenizer and prompt template

Our closed-vocabulary tokenizer knows ~60 words. That is enough for the simulated task and keeps the embedding table tiny. The prompt reserves one `<image>` placeholder per visual token, the same trick HF LLaVA/SmolVLM use:

`<bos> <image>×16  cmd: ... | at: ... | holding: ... | visited: ... <sep>`  →  model writes `pick red cup <eos>`

In [ ]:
tok = RobotTokenizer.from_domain()
print('vocab size:', len(tok))
prompt = 'cmd: bring me the red cup | at: kitchen | holding: nothing | visited: kitchen'
ids, labels = tok.encode_example(prompt, 'pick red cup', cfg.num_image_tokens)
print('sequence length:', len(ids))
print('tokens :', [tok.itos[i] for i in ids])
print('labels :', [tok.itos[l] if l >= 0 else '_' for l in labels])
print('-> loss is only computed on the answer (labels != -100)')

## Step 6: merge text and image embeddings

Embed the token ids, then **replace** each `<image>` slot with the next visual token. We use `cumsum` + `gather` + `where` instead of boolean indexing because it exports cleanly to ONNX/TensorRT (notebook 05).

In [ ]:
cfg = TinyVLMConfig(vocab_size=len(tok), image_token_id=tok.image_id, pad_token_id=tok.pad_id)
model = TinyVLM(cfg).eval()
input_ids = torch.tensor([ids])
with torch.no_grad():
    vt = model.encode_image(x)
    h = model.merge_embeddings(input_ids, vt)
print('merged sequence:', tuple(h.shape))
print('slot 1 is visual token 0:', torch.allclose(h[0, 1], vt[0, 0]))
print('slot 0 is the <bos> embedding:', torch.allclose(h[0, 0], model.embed.weight[tok.bos_id]))

## Step 7: the decoder LM (Llama-style)

Each block is `RMSNorm → grouped-query attention (with RoPE) → residual → RMSNorm → SwiGLU MLP → residual`.

* **Causal mask:** token *t* may only attend to tokens ≤ *t*. This makes next-token prediction trainable in parallel.
* **RoPE:** encodes position by *rotating* query/key pairs. Relative offsets become angle differences, and there is no learned position table.
* **GQA:** 6 query heads share 2 key/value heads, so the KV cache is 3x smaller. Qwen2.5-3B uses 16 Q heads with 2 KV heads (8x).

In [ ]:
T = 8
causal = torch.tril(torch.ones(T, T))
cos, sin = rope_tables(64, 32, 10000.0)
q = torch.randn(1, 1, 64, 32)
fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
ax[0].imshow(causal, cmap='Greens'); ax[0].set_title('causal mask (row attends to green)')
ax[0].set_xlabel('key position'); ax[0].set_ylabel('query position')
ax[1].plot(cos[:, 0], label='dim 0 (fast)'); ax[1].plot(cos[:, 14], label='dim 14 (slow)')
ax[1].set_title('RoPE: rotation angle vs position'); ax[1].legend(); plt.show()
print('RoPE keeps vector norms:', torch.allclose(apply_rope(q, cos, sin).norm(dim=-1), q.norm(dim=-1), atol=1e-5))

attn = model.layers[0].attn
print(f'Q heads={attn.h}, KV heads={attn.kvh}, head_dim={attn.hd}')

## Step 8: the whole model, parameters and an untrained forward pass

An untrained model should predict roughly uniformly, so the loss should be about $\ln(\text{vocab})$. Always check this sanity value before training.

In [ ]:
for k, v in model.param_breakdown().items():
    print(f'{k:20s} {v/1e6:6.3f} M')
labels_t = torch.tensor([labels])
with torch.no_grad():
    logits, loss = model(input_ids, x, labels_t)
print('\nlogits:', tuple(logits.shape), f' loss={loss.item():.3f}  ln(vocab)={math.log(len(tok)):.3f}')

## Step 9: generation and the KV cache

Without a cache, every new token re-processes the whole sequence, which is $O(T^2)$ work overall. With a cache we run the prompt once (**prefill**) and then feed **one** token per step (**decode**), re-using stored keys/values. The output is identical. Only the cost changes. On a Jetson this is the difference between usable and unusable.

In [ ]:
prompt_ids = torch.tensor([tok.encode_prompt(prompt, cfg.num_image_tokens)])
for use_cache in [False, True]:
    t = time.perf_counter()
    out = model.generate(prompt_ids, x, max_new_tokens=20, use_cache=use_cache)
    dt = (time.perf_counter() - t) * 1000
    print(f'use_cache={use_cache!s:5s} {dt:7.1f} ms  ->', repr(tok.decode(out.tolist())))
print('(untrained -> gibberish, but identical with and without cache)')

## How TinyVLM maps onto real small VLMs

| | TinyVLM (ours) | SmolVLM-256M | Qwen2.5-VL-3B |
|---|---|---|---|
| Vision encoder | 3-layer ViT, 0.6M | SigLIP-B/16, ~93M | ViT ~670M (window attn) |
| Image → tokens | 64 px → 64 patches → **16** | 512 px tile → 1024 → **64** (shuffle 4) | dynamic res, 2x2 merge |
| Connector | 2-layer MLP | linear | 2-layer MLP |
| LM | 4 layers, d=192, GQA 6/2 | SmolLM2-135M, 30 layers, GQA 9/3 | Qwen2.5-3B, 36 layers, GQA 16/2 |
| Vocab | ~60 words | 49k BPE | 151k BPE |

The structure is identical. Only the scale differs. In notebook 04 you will fine-tune SmolVLM-256M on the same robot data.

### Exercises
1. Set `pixel_shuffle=1` and `4` in `TinyVLMConfig`. How do `num_image_tokens` and the parameter count change?
2. Make `lm_kv_heads = lm_heads` (plain multi-head attention) and compare the KV-cache shape returned by `model.lm_forward`.
3. Time `generate` with `max_new_tokens=60` (raise `max_seq_len`). How does the cache speed-up grow with length?